In [ ]:
import os
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "project_paths.py").exists():
    candidate = PROJECT_ROOT.parent
    if (candidate / "project_paths.py").exists():
        PROJECT_ROOT = candidate

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

os.chdir(PROJECT_ROOT)


In [5]:
import pandas as pd

In [6]:
df=pd.read_excel("data/raw/Book1.xlsx")

FileNotFoundError: [Errno 2] No such file or directory: 'data/raw/Book1.xlsx'

In [ ]:
from sklearn.model_selection import train_test_split

# X = features, y = target
X = df.drop(columns=['severity'])
y = df['severity']

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,      # 20% test
    random_state=42,    # reproducibility
    stratify=y          # use this if classification
)

print(X_train.shape, X_test.shape)

(40000, 1) (10000, 1)


In [ ]:
import pandas as pd

# Define label mappings
unique_labels = sorted(y_train.unique().tolist())
label2id = {label: i for i, label in enumerate(unique_labels)}
id2label = {i: label for i, label in enumerate(unique_labels)}

# Convert y_train to numerical labels
y_train_encoded = y_train.map(label2id)

train_df = pd.DataFrame({
    "text": X_train['grievance_text'],
    "label": y_train_encoded # Use encoded labels
})

In [ ]:
from transformers import BertTokenizer

tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')

def tokenize(example):
    return tokenizer(
        example['text'],
        padding='max_length',
        truncation=True,
        max_length=128
    )

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

In [ ]:
from datasets import Dataset

train_dataset = Dataset.from_pandas(train_df)
train_dataset = train_dataset.map(tokenize, batched=True)

train_dataset = train_dataset.rename_column("label", "labels")
train_dataset.set_format(type='torch', columns=['input_ids', 'attention_mask', 'labels'])

Map:   0%|          | 0/40000 [00:00<?, ? examples/s]

In [ ]:
from transformers import BertForSequenceClassification

model = BertForSequenceClassification.from_pretrained(
    'bert-base-uncased',
    num_labels=len(unique_labels), # Use len(unique_labels) from earlier
    id2label=id2label,
    label2id=label2id
)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.bias                       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [ ]:
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="./results",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    num_train_epochs=3,
    weight_decay=0.01,
    logging_dir="./logs",
    logging_steps=10
)

`logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.


In [ ]:
from transformers import Trainer

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset
)

In [ ]:
trainer.train()

Step,Training Loss
10,1.552986
20,1.549791
30,1.498798
40,1.501251
50,1.486634
60,1.373147
70,1.253361
80,1.191013
90,1.088813
100,1.041383


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=7500, training_loss=0.8654086174647013, metrics={'train_runtime': 3285.5752, 'train_samples_per_second': 36.523, 'train_steps_per_second': 2.283, 'total_flos': 7893544273920000.0, 'train_loss': 0.8654086174647013, 'epoch': 3.0})

In [ ]:
# Encode y_test with numerical labels
y_test_encoded = y_test.map(label2id)

# Create a DataFrame for the test set
test_df = pd.DataFrame({
    "text": X_test['grievance_text'],
    "label": y_test_encoded
})

# Convert to Hugging Face Dataset
test_dataset = Dataset.from_pandas(test_df)

# Tokenize the test dataset
test_dataset = test_dataset.map(tokenize, batched=True)

# Rename the label column and set the format
test_dataset = test_dataset.rename_column("label", "labels")
test_dataset.set_format(type='torch', columns=['input_ids', 'attention_mask', 'labels'])

# Evaluate the model
results = trainer.evaluate(test_dataset)
print(results)

Map:   0%|          | 0/10000 [00:00<?, ? examples/s]

{'eval_loss': 0.844772458076477, 'eval_runtime': 66.8787, 'eval_samples_per_second': 149.525, 'eval_steps_per_second': 18.691, 'epoch': 3.0}


In [ ]:
from sklearn.metrics import accuracy_score
import numpy as np

# Make predictions on the test dataset
predictions_output = trainer.predict(test_dataset)

# The predictions_output.predictions are the raw logits
# We need to apply argmax to get the predicted labels
predicted_labels = np.argmax(predictions_output.predictions, axis=1)

# The true labels are in predictions_output.label_ids
true_labels = predictions_output.label_ids

# Calculate accuracy
accuracy = accuracy_score(true_labels, predicted_labels)

print(f"Model Accuracy: {accuracy}")

Model Accuracy: 0.4379


In [ ]:
from sklearn.metrics import f1_score

# Calculate F1-score
f1 = f1_score(true_labels, predicted_labels, average='weighted')

print(f"Model F1-score (weighted): {f1}")

Model F1-score (weighted): 0.39707244432139593
